# Recherche de produits et produits similaires

Un catalogue de 20 000 produits d'Amazon, 150 vraies requêtes de clients, et pour chacune des jugements humains de pertinence : E (*Exact*, le produit demandé), S (*Substitute*, un substitut acceptable), C (*Complement*, un complément, par exemple un étui pour un téléphone) ou I (*Irrelevant*). Le catalogue et les requêtes sont en anglais.

Le jeu de données tient dans trois fichiers Parquet, chargés plus bas dans `produits`, `requetes` et `qrels` :

- `products.parquet` : le catalogue, un produit par ligne. `product_id` (l'identifiant Amazon), `title`, `brand`, `color`, `bullets` (les points clés de la fiche) et `description` (nettoyés du HTML et coupés à 600 caractères). Certains champs sont vides : la couleur et la description le sont souvent. Le TP n'utilise que le titre, la marque et les points clés
- `queries.parquet` : les 150 requêtes, une par ligne. `query_id` et `query` (le texte tapé par le client, fautes de frappe comprises)
- `qrels.parquet` : les jugements de pertinence, une ligne par couple (requête, produit) jugé. `query_id`, `product_id` et `esci_label` (E, S, C ou I). Chaque requête a entre 8 et 43 produits jugés, 3 728 jugements en tout. Un produit sans jugement pour une requête compte comme non pertinent pour elle

1. Écrire les mesures d'évaluation
2. Une référence : BM25
3. La recherche dense, exacte puis approximative (HNSW)
4. La recherche hybride
5. Le reclassement par un cross-encodeur
6. Comparer, et lire les échecs
7. Des produits similaires

Ce notebook va plus vite avec un GPU : *Exécution > Modifier le type d'exécution > GPU T4*. Sans GPU, il tourne quand même : les deux calculs trop longs pour un processeur, encoder le catalogue (exercice 3) et reclasser 100 requêtes (exercice 5), n'y sont faits que sur un échantillon, puis remplacés par leur résultat calculé sur GPU.

## Données et modèles

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh esci-products-subset
%pip install -q bm25s faiss-cpu

In [ ]:
import os
import time

# bm25s se sert de JAX s'il est installé (c'est le cas sur Colab), et JAX réserve d'office
# 75 % de la mémoire du GPU : on le cantonne au CPU, le GPU est pour les modèles.
os.environ["JAX_PLATFORMS"] = "cpu"

import numpy as np
import pandas as pd
import torch
from sentence_transformers import CrossEncoder, SentenceTransformer

produits = pd.read_parquet("esci-products-subset/products.parquet")
requetes = pd.read_parquet("esci-products-subset/queries.parquet")
qrels = pd.read_parquet("esci-products-subset/qrels.parquet")
produits["texte"] = produits.title + ". " + produits.brand + ". " + produits.bullets
print(len(produits), "produits,", len(requetes), "requêtes,", len(qrels), "jugements")

GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
# Sur CPU, float32 : les demi-précisions (float16, et bfloat16, celle des poids publiés,
# qu'on aurait par défaut) y sont très lentes.
DTYPE = torch.float16 if GPU else torch.float32
print("GPU" if GPU else "Pas de GPU : les calculs longs chargeront leur résultat")

encodeur = SentenceTransformer(
  "Qwen/Qwen3-Embedding-0.6B", device=DEVICE, model_kwargs={"dtype": DTYPE}
)
encodeur.max_seq_length = (
  128  # le début d'une fiche produit suffit, et l'encodage va deux fois plus vite
)
reclasseur = CrossEncoder(
  "BAAI/bge-reranker-v2-m3",
  device=DEVICE,
  max_length=512,
  model_kwargs={"dtype": DTYPE},
)
print("Modèles chargés")

## 1. Écrire les mesures d'évaluation

Un moteur de recherche renvoie, pour chaque requête, une liste ordonnée d'identifiants de produits. On le mesure avec :

- le **rappel à 100** : la part des produits *Exact* de la requête qu'on trouve dans les 100 premiers résultats (ce que verra une étape de reclassement)
- le **nDCG à 10** : la qualité des 10 premiers résultats (ce que voit le client). Chaque résultat rapporte un gain selon son jugement (E : 1, S : 0,1, C : 0,01, I ou non jugé : 0), divisé par $\log_2(\text{rang} + 1)$ ; la somme (DCG) est divisée par celle du meilleur classement possible (DCG idéal)

Écrivez `rappel_a` et `ndcg_a`. `gains[id_requete]` associe à chaque produit jugé pour une requête son gain.

**Documentation utile**

- [Discounted cumulative gain](https://en.wikipedia.org/wiki/Discounted_cumulative_gain)
- [`numpy.log2`](https://numpy.org/doc/stable/reference/generated/numpy.log2.html)
- [`sorted`](https://docs.python.org/3/library/functions.html#sorted)

In [ ]:
GAIN = {"E": 1.0, "S": 0.1, "C": 0.01, "I": 0.0}
gains = {
  id_requete: dict(zip(groupe.product_id, groupe.esci_label.map(GAIN)))
  for id_requete, groupe in qrels.groupby("query_id")
}


def rappel_a(classement: list[str], gains_q: dict[str, float], k: int = 100) -> float:
  # Votre code ici
  ...


def ndcg_a(classement: list[str], gains_q: dict[str, float], k: int = 10) -> float:
  # Votre code ici
  ...

In [ ]:
def evalue(resultat: dict[int, list[str]]) -> pd.Series:
  """Le rappel@100 et le nDCG@10 moyens d'un classement : identifiant de requête -> identifiants de produits classés."""
  return pd.Series(
    {
      "rappel@100": np.mean([rappel_a(resultat[q], gains[q]) for q in resultat]),
      "nDCG@10": np.mean([ndcg_a(resultat[q], gains[q]) for q in resultat]),
    }
  )

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- les produits *Exact* d'une requête sont ceux dont le gain vaut `1.0`
- `gains_q.get(produit, 0.0)` : le gain d'un produit, 0 s'il n'a pas été jugé
- `np.log2(rang + 1)` : la décote ; les rangs commencent à 1 (`enumerate(classement[:k], start=1)`)
- le DCG idéal : la même somme sur `sorted(gains_q.values(), reverse=True)[:k]`
- pour vérifier, avec `exemple = {"a": 1.0, "b": 0.1, "c": 0.0, "d": 1.0}` (« a » et « d » sont *Exact*, « b » est *Substitute*) : `rappel_a(["a", "x", "d"], exemple)` vaut `1.0`, `rappel_a(["a", "x", "y"], exemple, k=2)` vaut `0.5`, `ndcg_a(["a", "d", "b"], exemple)` vaut `1.0` et `ndcg_a(["x", "a", "d"], exemple)` environ `0.673`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def rappel_a(classement: list[str], gains_q: dict[str, float], k: int = 100) -> float:
  # Les produits Exact de la requête : ceux dont le gain vaut 1.0
  exacts = {produit for produit, gain in gains_q.items() if ...}
  # Ceux qui figurent parmi les k premiers résultats
  trouves = exacts & set(classement[:...])
  return ...  # la part des produits Exact qu'on a trouvés


def ndcg_a(classement: list[str], gains_q: dict[str, float], k: int = 10) -> float:
  # DCG : chacun des k premiers résultats rapporte son gain (0 s'il n'a pas été jugé : gains_q.get(produit, 0.0)),
  # divisé par log2(rang + 1)
  dcg = sum(... for rang, produit in enumerate(classement[:k], start=1))
  # DCG idéal : la même somme sur le meilleur classement possible, c'est-à-dire les gains de la requête par ordre décroissant
  ideal = ...
  idcg = sum(... for rang, gain in enumerate(ideal, start=1))
  return ...
```

### Solution

In [ ]:
def rappel_a(classement: list[str], gains_q: dict[str, float], k: int = 100) -> float:
  exacts = {produit for produit, gain in gains_q.items() if gain == 1.0}
  return len(exacts & set(classement[:k])) / len(exacts)


def ndcg_a(classement: list[str], gains_q: dict[str, float], k: int = 10) -> float:
  dcg = sum(
    gains_q.get(produit, 0.0) / np.log2(rang + 1)
    for rang, produit in enumerate(classement[:k], start=1)
  )
  ideal = sorted(gains_q.values(), reverse=True)[:k]
  idcg = sum(gain / np.log2(rang + 1) for rang, gain in enumerate(ideal, start=1))
  return dcg / idcg

## 2. Une référence : BM25

[`bm25s`](https://github.com/xhluca/bm25s) indexe un corpus pour BM25 en quelques secondes. La cellule suivante définit `recherche_bm25(k1, b)`, qui indexe les textes des produits avec ces deux paramètres et renvoie les 100 meilleurs produits de chaque requête, sous la forme `{id_requete: [id_produit, ...]}`. Avec les valeurs par défaut de `bm25s`, $k_1 = 1{,}5$ et $b = 0{,}75$, elle donne `resultats["BM25"]`, la référence des exercices suivants.

Ces deux paramètres de la formule règlent la **saturation** ($k_1$ : plus il est petit, moins un mot répété dans une fiche rapporte) et la **longueur normalisée** ($b$, entre 0 et 1 : à 0, la longueur d'une fiche est ignorée ; à 1, elle pèse pleinement). Les fiches du catalogue vont de quelques mots à plus de 150. Comparez au nDCG@10 les valeurs par défaut, `b=0` (sans normalisation de longueur), `k1=0.5` et `k1=3`. Que règle chaque paramètre, et lequel compte le plus ici ?

In [ ]:
import bm25s

resultats = {}

# Les textes, tokenisés une fois pour toutes : seul l'index dépend de k1 et de b.
tokens_produits = bm25s.tokenize(
  produits.texte.tolist(), stopwords="en", show_progress=False
)
tokens_requetes = bm25s.tokenize(
  requetes["query"].tolist(), stopwords="en", show_progress=False
)


def recherche_bm25(k1: float = 1.5, b: float = 0.75) -> dict[int, list[str]]:
  """Les 100 meilleurs produits de chaque requête selon BM25 : identifiant de requête -> identifiants de produits."""
  moteur = bm25s.BM25(k1=k1, b=b)
  moteur.index(tokens_produits, show_progress=False)
  # `indices` : les positions des 100 meilleurs produits dans le catalogue, une ligne par requête
  indices, _ = moteur.retrieve(tokens_requetes, k=100, show_progress=False)
  return {
    id_requete: produits.product_id.iloc[ligne].tolist()
    for id_requete, ligne in zip(requetes.query_id, indices)
  }


resultats["BM25"] = recherche_bm25()  # par défaut dans bm25s : k1 = 1,5 et b = 0,75
print(evalue(resultats["BM25"]))

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `recherche_bm25(k1=..., b=...)` renvoie un classement, `{id_requete: [id_produit, ...]}`, que `evalue` mesure ; `evalue(resultat)["nDCG@10"]` en garde le nDCG@10
- un seul paramètre change à la fois, l'autre garde sa valeur par défaut : `(k1, b)` vaut `(1.5, 0.75)`, `(1.5, 0)`, `(0.5, 0.75)` puis `(3, 0.75)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
# Un seul paramètre change à la fois : les valeurs par défaut, b=0, k1=0.5 puis k1=3
for k1, b in [(1.5, 0.75), (1.5, 0), (0.5, 0.75), (3, 0.75)]:
  ndcg = ...  # le nDCG@10 du classement de BM25 avec ces deux paramètres
  print(f"k1={k1}, b={b} : nDCG@10 {ndcg:.3f}")
```

### Solution

In [ ]:
for k1, b in [(1.5, 0.75), (1.5, 0), (0.5, 0.75), (3, 0.75)]:
  ndcg = evalue(recherche_bm25(k1=k1, b=b))["nDCG@10"]
  print(f"k1={k1}, b={b} : nDCG@10 {ndcg:.3f}")

# k1 règle la saturation : plus il est petit, plus vite un mot répété cesse de rapporter.
# b règle la normalisation par la longueur : à 0, une fiche longue, qui contient plus de
# mots, passe devant une fiche courte aussi pertinente. Ici, b compte le plus : sans
# normalisation de longueur, le nDCG@10 tombe de 0,473 à 0,430, alors que k1 ne le fait
# varier que de 0,455 (k1=3) à 0,480 (k1=0.5). Une saturation rapide aide un peu : qu'un
# mot de la requête revienne souvent dans une fiche n'en dit guère plus sur elle.

## 3. La recherche dense

Encodez les produits (`encodeur.encode_document`) et les requêtes (`encodeur.encode_query`). Les vecteurs sont normalisés : le cosinus est un produit scalaire. Trouvez les 100 produits les plus proches de chaque requête par un produit de matrices et `np.argsort`, rangez le résultat dans `resultats["dense"]` et évaluez. Sans GPU, `textes_produits` et `textes_requetes` ne contiennent que 32 produits et 16 requêtes : l'exercice suivant charge les vecteurs de tous, calculés sur GPU, et refait ce classement.

**Documentation utile**

- [`numpy.matmul`](https://numpy.org/doc/stable/reference/generated/numpy.matmul.html)
- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
# Tous les produits et toutes les requêtes sur GPU ; sans GPU, 32 produits et 16 requêtes
# (une minute environ).
textes_produits = produits.texte.tolist() if GPU else produits.texte[:32].tolist()
textes_requetes = requetes["query"].tolist() if GPU else requetes["query"][:16].tolist()

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `encodeur.encode_document(textes, batch_size=32, show_progress_bar=True)` pour les produits (`textes_produits`), `encodeur.encode_query(textes)` pour les requêtes (`textes_requetes`)
- `.astype(np.float32)` : sur GPU, le modèle est en demi-précision ; on calcule en `float32`
- `vecteurs_requetes @ vecteurs_produits.T` : les scores de toutes les requêtes contre tous les produits d'un coup, une ligne par requête
- `np.argsort(-scores, axis=1)[:, :100]` : les positions des 100 meilleurs produits de chaque requête (on trie les scores opposés pour avoir les meilleurs en tête)
- `produits.product_id.iloc[ligne].tolist()` : les identifiants des produits à ces positions

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
debut = time.time()
# Les textes des produits (`textes_produits`) et ceux des requêtes (`textes_requetes`)
vecteurs_produits = encodeur.encode_document(
  ..., batch_size=32, show_progress_bar=True
).astype(np.float32)
vecteurs_requetes = encodeur.encode_query(...).astype(np.float32)
print(f"{time.time() - debut:.0f} s pour encoder le catalogue")

# Une ligne par requête, une colonne par produit
scores = ...
# `meilleurs` : pour chaque requête, les positions de ses 100 meilleurs produits
meilleurs = np.argsort(..., axis=1)[:, :100]
resultats["dense"] = {
  id_requete: ...  # les identifiants des produits aux positions `ligne`, sous forme de liste
  for id_requete, ligne in zip(requetes.query_id, meilleurs)
}
evalue(resultats["dense"])
```

### Solution

In [ ]:
debut = time.time()
vecteurs_produits = encodeur.encode_document(
  textes_produits, batch_size=32, show_progress_bar=True
).astype(np.float32)
vecteurs_requetes = encodeur.encode_query(textes_requetes).astype(np.float32)
print(f"{time.time() - debut:.0f} s pour encoder le catalogue")

scores = vecteurs_requetes @ vecteurs_produits.T
meilleurs = np.argsort(-scores, axis=1)[:, :100]
resultats["dense"] = {
  id_requete: produits.product_id.iloc[ligne].tolist()
  for id_requete, ligne in zip(requetes.query_id, meilleurs)
}
evalue(resultats["dense"])

### L'index HNSW

Comparer une requête à tous les produits est exact, mais coûte trop cher à des millions de produits. Un index **HNSW** (un graphe de voisinage à plusieurs niveaux) trouve les plus proches voisins **approximativement**, beaucoup plus vite.

Construisez un index `faiss.IndexHNSWFlat`, ajoutez-y les vecteurs des produits, et cherchez les 100 plus proches voisins des requêtes. Mesurez le temps de recherche face à la recherche exacte, faite avec le même outil (`faiss.IndexFlatIP`, qui compare chaque requête à tous les produits), et la part des 100 voisins exacts que HNSW retrouve. Faites varier `index.hnsw.efSearch` : qu'est-ce que ce paramètre règle ?

**Documentation utile**

- [Faiss : premiers pas](https://github.com/facebookresearch/faiss/wiki/Getting-started)
- [Faiss : les index](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes)
- [`time.time`](https://docs.python.org/3/library/time.html#time.time)

In [ ]:
# Sans GPU, l'exercice précédent n'a encodé qu'une partie du catalogue : on charge les
# vecteurs de tous les produits et de toutes les requêtes, calculés sur GPU par le même
# code (stockés en float16), et on refait le classement dense.
if not GPU:
  vecteurs_produits = np.load(
    "esci-products-subset/lab-results/product-embeddings.npy"
  ).astype(np.float32)
  vecteurs_requetes = np.load(
    "esci-products-subset/lab-results/query-embeddings.npy"
  ).astype(np.float32)
  meilleurs = np.argsort(-(vecteurs_requetes @ vecteurs_produits.T), axis=1)[:, :100]
  resultats["dense"] = {
    id_requete: produits.product_id.iloc[ligne].tolist()
    for id_requete, ligne in zip(requetes.query_id, meilleurs)
  }
  print(evalue(resultats["dense"]))

# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `faiss.IndexHNSWFlat(dimension, 32, faiss.METRIC_INNER_PRODUCT)` : `dimension` est la taille des vecteurs (`vecteurs_produits.shape[1]`), 32 le nombre de voisins auxquels chaque nœud du graphe est relié, et la mesure est le produit scalaire (les vecteurs sont normalisés)
- `index.add(vecteurs)`, puis `index.search(vecteurs_requêtes, 100)`, qui renvoie les scores et les positions des 100 voisins de chaque requête
- `index.hnsw.efSearch` : essayez `16`, `64` et `128`
- `faiss.IndexFlatIP(dimension)` : la recherche exacte, avec les mêmes `add` et `search` que l'index HNSW
- `time.time()` avant et après pour chronométrer
- la part des voisins exacts retrouvés : l'intersection entre les 100 positions de HNSW et celles de `meilleurs` (la recherche exacte), divisée par 100, en moyenne sur les requêtes

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
import faiss

# 32 est le nombre de voisins auxquels chaque nœud du graphe est relié
index = faiss.IndexHNSWFlat(
  ..., 32, faiss.METRIC_INNER_PRODUCT
)  # le premier argument : la dimension des vecteurs
index.add(...)  # les vecteurs des produits

for ef in (16, 64, 128):
  index.hnsw.efSearch = ef
  debut = time.time()
  _, approches = index.search(
    ..., 100
  )  # `approches` : les positions des 100 voisins de chaque requête
  duree = time.time() - debut
  # La part des 100 voisins exacts (`meilleurs`, de la recherche exacte) que HNSW retrouve aussi, en moyenne sur les requêtes
  part = ...
  print(f"efSearch {ef:3} : {duree * 1000:.0f} ms, {part:.1%} des voisins exacts")

# La recherche exacte, avec le même outil, pour comparer
exacte = faiss.IndexFlatIP(...)  # la dimension des vecteurs
exacte.add(...)  # les vecteurs des produits
debut = time.time()
...  # la recherche des 100 plus proches voisins des requêtes
print(f"recherche exacte : {(time.time() - debut) * 1000:.0f} ms")
```

#### Solution

In [ ]:
import faiss

index = faiss.IndexHNSWFlat(vecteurs_produits.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
index.add(vecteurs_produits)
for ef in (16, 64, 128):
  index.hnsw.efSearch = ef
  debut = time.time()
  _, approches = index.search(vecteurs_requetes, 100)
  duree = time.time() - debut
  part = np.mean([len(set(a) & set(b)) / 100 for a, b in zip(approches, meilleurs)])
  print(f"efSearch {ef:3} : {duree * 1000:.0f} ms, {part:.1%} des voisins exacts")

exacte = faiss.IndexFlatIP(vecteurs_produits.shape[1])
exacte.add(vecteurs_produits)
debut = time.time()
exacte.search(vecteurs_requetes, 100)
print(f"recherche exacte : {(time.time() - debut) * 1000:.0f} ms")

# efSearch est la taille de la liste de candidats explorée dans le graphe : plus grand,
# plus juste et plus lent. Sur 20 000 produits, la recherche exacte reste rapide, à peine
# deux fois plus lente que HNSW à efSearch 128 ; HNSW devient indispensable à des
# millions de vecteurs. Un produit de matrices suivi d'un np.argsort l'aurait fait
# paraître plusieurs fois plus lente : trier les 20 000 scores de chaque requête coûte
# plus que les calculer.

## 4. La recherche hybride

La fusion par les rangs (*Reciprocal Rank Fusion*, RRF) combine plusieurs classements sans avoir à comparer leurs scores : chaque produit reçoit $\sum 1 / (60 + \text{rang})$ sur les classements où il apparaît (rang à partir de 1). Écrivez `rrf(*classements)`, fusionnez BM25 et la recherche dense pour chaque requête dans `resultats["hybride"]` (gardez les 100 premiers), et évaluez.

**Documentation utile**

- [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf)
- [`collections.defaultdict`](https://docs.python.org/3/library/collections.html#collections.defaultdict)
- [`enumerate`](https://docs.python.org/3/library/functions.html#enumerate)
- [`sorted`](https://docs.python.org/3/library/functions.html#sorted)

In [ ]:
def rrf(*classements: list[str], k: int = 60) -> list[str]:
  # Votre code ici
  ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `rrf(*classements, k=60)` : chaque produit reçoit `1 / (k + rang)` pour chaque classement où il apparaît ; les rangs commencent à 1 (`enumerate(classement, start=1)`)
- `defaultdict(float)` : un score de 0 pour un produit pas encore vu
- `sorted(scores, key=scores.get, reverse=True)` : les produits par score décroissant
- `rrf(resultats["BM25"][id_requete], resultats["dense"][id_requete])[:100]` pour chaque requête
- pour vérifier : `rrf(["a", "b", "c"], ["b", "c", "d"])` renvoie `["b", "c", "a", "d"]`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from collections import defaultdict


def rrf(*classements: list[str], k: int = 60) -> list[str]:
  scores = defaultdict(float)
  for classement in classements:
    for rang, produit in enumerate(classement, start=1):
      scores[produit] += ...  # la contribution de ce classement : 1 / (k + rang)
  return sorted(scores, key=..., reverse=True)  # les produits par score décroissant


resultats["hybride"] = {
  id_requete: rrf(..., ...)[:100]  # les classements BM25 et dense de cette requête
  for id_requete in requetes.query_id
}
evalue(resultats["hybride"])
```

### Solution

In [ ]:
from collections import defaultdict


def rrf(*classements: list[str], k: int = 60) -> list[str]:
  scores = defaultdict(float)
  for classement in classements:
    for rang, produit in enumerate(classement, start=1):
      scores[produit] += 1 / (k + rang)
  return sorted(scores, key=scores.get, reverse=True)


resultats["hybride"] = {
  id_requete: rrf(resultats["BM25"][id_requete], resultats["dense"][id_requete])[:100]
  for id_requete in requetes.query_id
}
evalue(resultats["hybride"])

# La fusion trouve plus de produits exacts dans ses 100 premiers (rappel@100) : c'est ce
# qu'exploite l'étape suivante, le reclassement. Son nDCG@10 n'est pas forcément meilleur
# que celui de la recherche dense quand BM25 est nettement plus faible.

## 5. Le reclassement par un cross-encodeur

Un cross-encodeur lit la requête et le produit **ensemble** : plus juste, mais une passe du modèle par paire. Pour les 100 premières requêtes (`lot` ; sans GPU, la première seulement : l'exercice suivant charge le reclassement des 100, calculé sur GPU), reclassez les 50 premiers résultats hybrides avec le cross-encodeur `reclasseur`, puis ajoutez derrière les 50 suivants dans l'ordre hybride. Rangez le résultat dans `resultats["hybride + reclassement"]` et comparez avec l'hybride sur les **mêmes** 100 requêtes.

**Documentation utile**

- [`CrossEncoder.predict`](https://sbert.net/docs/package_reference/cross_encoder/cross_encoder.html#sentence_transformers.cross_encoder.model.CrossEncoder.predict)
- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
texte_de = dict(zip(produits.product_id, produits.texte))
requete_de = dict(zip(requetes.query_id, requetes["query"]))
# Les 100 premières requêtes ; sans GPU, 1 (deux minutes environ).
lot = requetes.query_id.iloc[:100] if GPU else requetes.query_id.iloc[:1]

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `reclasseur.predict([(requête, texte), ...], batch_size=32)` : une paire (texte de la requête, texte du produit) par candidat, d'après `requete_de[id_requete]` et `texte_de[id_produit]` ; renvoie un tableau de scores
- `np.argsort(-scores)` : les positions des candidats par score décroissant
- `evalue({q: resultats["hybride"][q] for q in lot})` : l'hybride sur les mêmes 100 requêtes

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
debut = time.time()
reclasses = {}
for id_requete in lot:
  candidats = resultats["hybride"][id_requete][:50]
  # Le cross-encodeur note une paire (texte de la requête, texte du produit) par candidat
  scores = reclasseur.predict([(..., ...) for p in candidats], batch_size=32)
  # Les candidats par score décroissant (`np.argsort` renvoie des positions)
  ordre = [candidats[i] for i in ...]
  reclasses[
    id_requete
  ] = ...  # les 50 reclassés, puis les 50 suivants du classement hybride
resultats["hybride + reclassement"] = reclasses
print(f"{time.time() - debut:.0f} s pour reclasser {len(lot)} requêtes")

# Comparaison avec l'hybride sur les mêmes 100 requêtes
pd.DataFrame(
  {
    "hybride": ...,
    "hybride + reclassement": ...,
  }
)
```

### Solution

In [ ]:
debut = time.time()
reclasses = {}
for id_requete in lot:
  candidats = resultats["hybride"][id_requete][:50]
  scores = reclasseur.predict(
    [(requete_de[id_requete], texte_de[p]) for p in candidats], batch_size=32
  )
  ordre = [candidats[i] for i in np.argsort(-scores)]
  reclasses[id_requete] = ordre + resultats["hybride"][id_requete][50:]
resultats["hybride + reclassement"] = reclasses
print(f"{time.time() - debut:.0f} s pour reclasser {len(lot)} requêtes")

pd.DataFrame(
  {
    "hybride": evalue({q: resultats["hybride"][q] for q in lot}),
    "hybride + reclassement": evalue(reclasses),
  }
)

## 6. Comparer, et lire les échecs

Rassemblez dans un tableau les mesures de toutes les méthodes, sur les 100 requêtes reclassées, et ajoutez-y, pour chaque méthode, la part des 10 premiers résultats qui n'ont aucun jugement. Ces produits comptent comme non pertinents : que vaut alors un nDCG de 0,5 ? Puis cherchez les requêtes où BM25 bat nettement la recherche dense (en nDCG@10), et l'inverse : lisez-les. Qu'ont-elles en commun ?

**Documentation utile**

- [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [`DataFrame.nlargest`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nlargest.html) et [`nsmallest`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nsmallest.html)

In [ ]:
# Sans GPU, l'exercice précédent n'a reclassé qu'une requête : on charge le reclassement des
# 100, calculé sur GPU par le même code (les produits de chaque requête, dans l'ordre).
if not GPU:
  reclasses = pd.read_parquet("esci-products-subset/lab-results/reranked.parquet")
  resultats["hybride + reclassement"] = (
    reclasses.groupby("query_id").product_id.agg(list).to_dict()
  )
  lot = requetes.query_id.iloc[:100]

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `evalue` prend `{id_requete: classement}` : `{q: resultat[q] for q in lot}` restreint un résultat aux 100 requêtes
- `pd.DataFrame({nom: série, ...})` : une colonne par méthode ; les méthodes sont les clés de `resultats`
- `p not in gains[q]` : le produit `p` n'a pas été jugé pour la requête `q` ; `np.mean` d'une liste de booléens donne la part de `True`
- `table.loc["non jugés@10"] = [...]` ajoute une ligne au tableau, une valeur par méthode, dans l'ordre de `resultats`
- `ndcg_a(resultats["BM25"][q], gains[q])` pour chaque requête, et de même pour `resultats["dense"]`
- `.nsmallest(8, "écart")` : les 8 plus petits écarts (BM25 gagne), `.nlargest(8, "écart")` les 8 plus grands (la recherche dense gagne)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
# Une colonne par méthode, chacune évaluée sur les mêmes 100 requêtes (`lot`) : `evalue` prend {id_requete: classement}
table = pd.DataFrame(
  {nom: evalue({q: ... for q in lot}) for nom, resultat in resultats.items()}
)
# La part des 10 premiers résultats sans jugement, en moyenne sur les requêtes : une valeur par méthode
table.loc["non jugés@10"] = [
  np.mean([np.mean([p not in gains[q] for p in ...]) for q in lot])
  for resultat in resultats.values()
]
print(table.round(3))

# nDCG@10 de BM25 et de la recherche dense, requête par requête
par_requete = pd.DataFrame(
  {
    "requete": [requete_de[q] for q in requetes.query_id],
    "BM25": [ndcg_a(..., gains[q]) for q in requetes.query_id],
    "dense": [ndcg_a(..., gains[q]) for q in requetes.query_id],
  }
)
par_requete[
  "écart"
] = ...  # positif quand la recherche dense gagne, négatif quand BM25 gagne
print("BM25 gagne :", par_requete.nsmallest(8, "écart")["requete"].tolist())
print("Le dense gagne :", ...)  # les 8 requêtes aux plus grands écarts
```

### Solution

In [ ]:
table = pd.DataFrame(
  {nom: evalue({q: resultat[q] for q in lot}) for nom, resultat in resultats.items()}
)
table.loc["non jugés@10"] = [
  np.mean([np.mean([p not in gains[q] for p in resultat[q][:10]]) for q in lot])
  for resultat in resultats.values()
]
print(table.round(3))

par_requete = pd.DataFrame(
  {
    "requete": [requete_de[q] for q in requetes.query_id],
    "BM25": [ndcg_a(resultats["BM25"][q], gains[q]) for q in requetes.query_id],
    "dense": [ndcg_a(resultats["dense"][q], gains[q]) for q in requetes.query_id],
  }
)
par_requete["écart"] = par_requete.dense - par_requete.BM25
print("BM25 gagne :", par_requete.nsmallest(8, "écart")["requete"].tolist())
print("Le dense gagne :", par_requete.nlargest(8, "écart")["requete"].tolist())

# Un tiers environ des 10 premiers résultats n'a aucun jugement, de 29 % après
# reclassement à 39 % pour BM25. Ils comptent pour 0 : le DCG est un minorant, le
# nDCG le plus souvent sous-estimé (pas toujours, car juger un produit fait aussi
# grandir le DCG idéal). Un nDCG de 0,5 peut cacher des produits pertinents que
# personne n'a jugés, et deux méthodes se comparent mal quand leurs parts de non-jugés
# diffèrent.
# BM25 gagne quand les mots de la requête sont ceux de la fiche, noms propres compris ;
# la recherche dense sur les fautes, les autres langues et les reformulations. Il y a des
# exceptions : la recherche dense gagne aussi sur une marque (mcm shades men) et sur un
# nom de produit collé (happypills, « Happy Pills » au catalogue).

## 7. Des produits similaires

Pour la page d'un produit, on propose des produits similaires : ses plus proches voisins dans l'espace des *embeddings*, sans requête. Écrivez `similaires(id_produit, k=5, meme_marque=False, score_min=-np.inf, score_max=np.inf)` qui affiche le score et le titre des `k` voisins d'un produit (sans le produit lui-même), éventuellement restreints à la même marque, dont le score est compris entre `score_min` et `score_max`. Essayez-la sur trois produits de `candidats` (des produits dont la marque a d'autres produits au catalogue), avec et sans `meme_marque`, et lisez les voisins : que proposeriez-vous au client pour ces trois produits ? Choisissez ensuite, d'après les scores que vous lisez, un score minimal, sous lequel un voisin n'a plus grand-chose à voir avec le produit, et un score maximal, au-dessus duquel c'est une variante du produit lui-même (une autre taille, une autre couleur), et relancez.

Et puisque le modèle est multilingue : cherchez dans le catalogue anglais avec des requêtes en français (« casque de moto avec écouteurs », « gamelle pour chien surélevée »).

**Documentation utile**

- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)
- [`numpy.inf`](https://numpy.org/doc/stable/reference/constants.html#numpy.inf)

In [ ]:
position = {
  produit: i for i, produit in enumerate(produits.product_id)
}  # identifiant du produit -> ligne du catalogue

# Des produits dont la marque a d'autres produits au catalogue.
marques = produits.brand[produits.brand != ""].value_counts()
candidats = produits[produits.brand.isin(marques[marques >= 5].index)]


def similaires(
  id_produit: str,
  k: int = 5,
  meme_marque: bool = False,
  score_min: float = -np.inf,
  score_max: float = np.inf,
) -> None:
  # Votre code ici
  ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `vecteurs_produits @ vecteurs_produits[i]` : la similarité de tous les produits avec le produit `i`
- `scores[i] = -np.inf` : un produit n'est pas son propre voisin
- `produits.brand.to_numpy() != produits.brand[i]` vaut `True` pour les produits d'une autre marque : donnez-leur aussi `-np.inf`
- `(scores < score_min) | (scores > score_max)` vaut `True` pour les voisins hors des deux seuils : donnez-leur aussi `-np.inf`
- `np.argsort(-scores)[:k]` : les positions des `k` meilleurs ; `np.isfinite(scores[j])` écarte les produits exclus (il peut en rester moins de `k`)
- `candidats.product_id.sample(3, random_state=0)` : trois produits de `candidats`
- des seuils à essayer : `score_min=0.5` et `score_max=0.98`
- pour une requête en français : `vecteurs_produits @ encodeur.encode_query(requête).astype(np.float32)`, puis `produits.title.iloc[positions]`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def similaires(
  id_produit: str,
  k: int = 5,
  meme_marque: bool = False,
  score_min: float = -np.inf,
  score_max: float = np.inf,
) -> None:
  i = position[id_produit]
  scores = vecteurs_produits @ vecteurs_produits[i]  # la similarité à tous les produits
  scores[i] = ...  # un produit n'est pas son propre voisin
  if meme_marque:
    # `produits.brand.to_numpy() != produits.brand[i]` vaut True pour les produits d'une autre marque
    scores[...] = -np.inf
  # Les voisins trop différents (sous score_min) et les variantes du produit (au-dessus de score_max)
  scores[...] = -np.inf
  print("Produit :", produits.title[i], "| marque :", produits.brand[i])
  for j in ...:  # les positions des k meilleurs scores
    if np.isfinite(scores[j]):  # il peut rester moins de k produits
      print(f"  {scores[j]:.3f} {produits.title[j][:100]}")


for id_produit in candidats.product_id.sample(3, random_state=0):
  ...  # les produits similaires, ceux de la même marque, puis avec les deux seuils

# Des requêtes en français, face au catalogue anglais
for requete in ("casque de moto avec écouteurs", "gamelle pour chien surélevée"):
  scores = (
    vecteurs_produits @ ...
  )  # le vecteur de la requête : encodeur.encode_query(...), en float32
  print(
    requete, "->", produits.title.iloc[...].str[:80].tolist()
  )  # les titres des 3 meilleurs produits
```

### Solution

In [ ]:
def similaires(
  id_produit: str,
  k: int = 5,
  meme_marque: bool = False,
  score_min: float = -np.inf,
  score_max: float = np.inf,
) -> None:
  i = position[id_produit]
  scores = vecteurs_produits @ vecteurs_produits[i]
  scores[i] = -np.inf
  if meme_marque:
    scores[produits.brand.to_numpy() != produits.brand[i]] = -np.inf
  scores[(scores < score_min) | (scores > score_max)] = -np.inf
  print("Produit :", produits.title[i], "| marque :", produits.brand[i])
  for j in np.argsort(-scores)[:k]:
    if np.isfinite(scores[j]):
      print(f"  {scores[j]:.3f} {produits.title[j][:100]}")


for id_produit in candidats.product_id.sample(3, random_state=0):
  similaires(id_produit)
  similaires(id_produit, meme_marque=True)
  similaires(id_produit, meme_marque=True, score_min=0.5, score_max=0.98)

# Le premier voisin de la chemise de nuit Ekouaer (0,998) est la même, dans une autre
# taille : une variante, que le score maximal écarte (au-dessus de 0,98, ce sont le plus
# souvent d'autres tailles, couleurs ou lots d'un même produit). Restreinte à la marque
# BestLand, la palette n'a que quatre voisins proches, et le cinquième, un fixateur pour
# cheveux à 0,359, n'a rien à voir : le score minimal l'écarte, quitte à proposer moins
# de k produits. Même marque ou non dépend de qui vend : une marque propose sa gamme (les
# engrais Advanced Nutrients), une place de marché aussi les produits concurrents (les
# palettes d'autres marques, un engrais House & Garden).

for requete in ("casque de moto avec écouteurs", "gamelle pour chien surélevée"):
  scores = vecteurs_produits @ encodeur.encode_query(requete).astype(np.float32)
  print(requete, "->", produits.title.iloc[np.argsort(-scores)[:3]].str[:80].tolist())